En esta práctica aprenderemos a ajustar el modelo GLM de Poisson en R a través de distintos ejemplos que consideran particularidades como el uso de offset, la sobredispersión, el exceso de ceros o la interacción entre predictores. En ellos se trabajará sobre la interpretación del modelo, su validez, inferencias sobre los parámetros y detección de puntos influyentes y/o atípicos. El objetivo es que el estudiante adquiera un dominio básico para abordar problemas similares en la práctica.

In [2]:
# Cargar librerías necesarias
library(ggplot2)
library(boot)
library(statmod)
library(car)
library(glmtoolbox)

: [1m[33mError[39m in `library()` at ]8;line = 6:col = 1;file://file:///tmp/ark-debug-16378/412998634.rtmp/ark-debug-16378/412998634.r:6:1]8;;:[22m
[33m![39m there is no package called 'glmtoolbox'

In [ ]:
# Función para generar los datos
gen_dat <- function(n, b0, b1) {
  x <- runif(n=n, min=0, max=10)  # Covariable x ~ Uniforme(0,1)
  lambda <- exp(b0 + b1 * x)  # Relación log-lineal con Poisson
  y <- rpois(n=n, lambda=lambda)  # Respuesta Poisson
  data.frame(y=y, x=x)
}

# Generando datos para n = 150
n <- 150
datos <- gen_dat(n=n, b0=-1, b1=0.3)

# Visualizar los primeros datos
head(datos)

In [ ]:
# Gráfico de dispersión de los datos simulados
ggplot(datos, aes(x = x, y = y)) +
  geom_point(alpha = 0.6, color = "blue") +  # Datos originales en azul
  labs(title = "Datos simulados: y ~ Poisson(lambda)",
       x = "Covariable x", 
       y = "Número de eventos (y)") +
  theme_minimal()

In [ ]:
# Ajustamos un modelo GLM con familia Poisson
model_pois <- glm(y ~ x, family = poisson(link="log"), data = datos)

# Resumen del modelo
summary(model_pois)

In [3]:
# Extraemos los coeficientes del modelo
coef_ajustados <- coef(model_pois)
coef_exponenciales <- exp(coef_ajustados)  # Exponencial de los coeficientes

# Tabla de comparación
coef_comparacion <- data.frame(
  Parámetro = c("Intercept", "Pendiente"),
  Simulado = c(-1, 0.3),
  Estimado = coef_ajustados,
  `Exp(coef)` = coef_exponenciales
)
coef_comparacion

: [1m[33mError[39m:[22m
[33m![39m Syntax error: unexpected '>'

En un modelo de Poisson, los coeficientes NO representan diferencias absolutas, sino diferencias en el logaritmo de la tasa esperada de eventos. Para interpretarlos, aplicamos la función exponencial 

El Intercept representa la tasa de incidencia esperada cuando x = 0.
La pendiente (exp(beta1)) es la razón de tasas de incidencia a veces denominada IRR.
Si beta1=0.29, entonces exp(0.3)=1.33, lo que significa que por cada unidad adicional en x, la tasa esperada de y aumenta en un 33%.

In [ ]:
# Predicciones del modelo
datos$pred <- predict(model_pois, type = "response")

# Graficamos los valores observados vs. ajustados
ggplot(datos, aes(x = x, y = y)) +
  geom_point(alpha = 0.5, color = "blue") +
  geom_line(aes(y = pred), color = "red", size = 1) +
  labs(title = "Valores Observados vs. Ajustados",
       x = "Variable Predictora (x)",
       y = "Conteo de Eventos (y)",
       subtitle = "Los valores predichos (línea roja) siguen la tendencia de los datos") +
  theme_minimal()

In [ ]:
# Calcular residuos
datos$res_pearson  <- boot::glm.diag(model_pois)$rp   # Residuos de Pearson
datos$res_deviance <- boot::glm.diag(model_pois)$rd   # Residuos de Deviance
datos$res_quantile <- statmod::qresid(model_pois)     # Residuos Cuantiles

# Ver primeras filas con residuos
head(datos)

In [ ]:
# a) Dispersión de los residuos en función de x 
ggplot(datos, aes(x = x)) +
  geom_point(aes(y = res_pearson), color = "blue", alpha = 0.6) +
  geom_point(aes(y = res_deviance), color = "red", alpha = 0.6) +
  geom_point(aes(y = res_quantile), color = "green", alpha = 0.6) +
  labs(title = "Comparación de Residuos: Pearson (azul), Deviance (rojo) y Cuantiles (verde)",
       x = "Covariable x", y = "Residuos") +
  theme_minimal()

In [ ]:
#Los residuos de Pearson  pueden ser más grandes en valores extremos.
#Los residuos de Deviance  suelen estar más ajustados en la región central.
#Los residuos Cuantiles  están más distribuidos en torno a 0 y deberían parecer normalizados.

In [ ]:
# b) QQ plots para evaluar normalidad
par(mfrow = c(1, 3))  
qqPlot(datos$res_pearson, dist = "norm", mean = 0, sd = 1, main = "QQ-Plot: Residuos Pearson")

qqPlot(datos$res_deviance, dist = "norm", mean = 0, sd = 1, main = "QQ-Plot: Residuos Deviance")

qqPlot(datos$res_quantile, dist = "norm", mean = 0, sd = 1, main = "QQ-Plot: Residuos Cuantiles")

In [ ]:
#Residuos de Pearson y Deviance pueden mostrar más valores atípicos.
#Residuos Cuantiles deben seguir mejor la normalidad.


# c) Comparación de histogramas de residuos
par(mfrow = c(1, 3)) 
hist(datos$res_pearson, main = "Histograma: Residuos Pearson", col = "blue", breaks = 20)
hist(datos$res_deviance, main = "Histograma: Residuos Deviance", col = "red", breaks = 20)
hist(datos$res_quantile, main = "Histograma: Residuos Cuantiles", col = "green", breaks = 20)
